# einops-rearrange — ex9: edge case: patch_size that doesn't divide H/W

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. Running the final beacon cell reports progress against the `Einops: Rearrange` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.rearrange — quick refresher

`rearrange(tensor, pattern, **axes_lengths)` is one operator with three jobs: **reorder** axes (`'h w -> w h'`), **compose** them (`'h w c -> (h w) c'`), and **decompose** them (`'(b1 b2) c -> b1 b2 c'`, with `b1=` or `b2=`). Every identifier on the right must appear on the left and vice versa.

The exercises below build on that: each one runs `rearrange` inside a small pipeline where you have to *see* what the layout did — by plotting it, by printing the shape at each step, or by combining 2–3 patterns into a single ML-adjacent transformation.

### Exercise 9 — edge case: patch_size that doesn't divide H/W

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Discover by execution that rearrange decomposition requires exact divisibility, and write a pad-then-patchify helper that handles non-divisible spatial dimensions cleanly.
> Keywords: edge-case, error-handling, padding, ViT
> ```

**KCs targeted:** `rearrange-axis-decomposition`

`rearrange` will *raise* on `'n c (h p1) (w p2) -> ...'` if `H` isn't a multiple of `p1` — there's no silent truncation. In real ViT-style code with non-square images (segmentation masks, medical slices, web crops), you usually want to pad up to the nearest multiple of `p` first.

Implement two things:

**`ex9_observe_failure(img, p)`** — call `rearrange(img, 'n c (h p1) (w p2) -> n (h w) (c p1 p2)', p1=p, p2=p)` inside a `try / except` and return the exception's message as a string (or `None` if no exception was raised). This makes the failure mode concrete instead of abstract.

**`ex9_safe_patchify(img, p)`** — pad `img` of shape `(N, C, H, W)` with **zeros** on the bottom and right so the new H and W are the smallest multiples of `p` that are ≥ the originals, *then* patchify. Return the tuple `(patches, padded_h, padded_w)` so the caller knows how to unpatchify back. Use `torch.nn.functional.pad` or direct tensor assignment for the pad; use `rearrange` for the patchify.

If the input H and W already divide `p`, `safe_patchify` should skip padding (return the original spatial dims).

In [ ]:
def ex9_observe_failure(img: Tensor, p: int) -> str | None:
    try:
        rearrange(img, 'n c (h p1) (w p2) -> n (h w) (c p1 p2)', p1=p, p2=p)
        return None
    except Exception as e:
        return f'{type(e).__name__}: {e}'


def ex9_safe_patchify(img: Tensor, p: int) -> tuple[Tensor, int, int]:
    import torch.nn.functional as F
    N, C, H, W = img.shape
    H_pad = ((H + p - 1) // p) * p
    W_pad = ((W + p - 1) // p) * p
    pad_h = H_pad - H
    pad_w = W_pad - W
    # F.pad takes pads in reverse-axis order: (left, right, top, bottom, ...)
    img_padded = F.pad(img, (0, pad_w, 0, pad_h), value=0.0) if (pad_h or pad_w) else img
    patches = rearrange(
        img_padded, 'n c (h p1) (w p2) -> n (h w) (c p1 p2)',
        p1=p, p2=p,
    )
    return patches, H_pad, W_pad


<details><summary>Solution</summary>

```python
def ex9_observe_failure(img: Tensor, p: int) -> str | None:
    try:
        rearrange(img, 'n c (h p1) (w p2) -> n (h w) (c p1 p2)', p1=p, p2=p)
        return None
    except Exception as e:
        return f'{type(e).__name__}: {e}'


def ex9_safe_patchify(img: Tensor, p: int) -> tuple[Tensor, int, int]:
    import torch.nn.functional as F
    N, C, H, W = img.shape
    H_pad = ((H + p - 1) // p) * p
    W_pad = ((W + p - 1) // p) * p
    pad_h = H_pad - H
    pad_w = W_pad - W
    # F.pad takes pads in reverse-axis order: (left, right, top, bottom, ...)
    img_padded = F.pad(img, (0, pad_w, 0, pad_h), value=0.0) if (pad_h or pad_w) else img
    patches = rearrange(
        img_padded, 'n c (h p1) (w p2) -> n (h w) (c p1 p2)',
        p1=p, p2=p,
    )
    return patches, H_pad, W_pad
```

**Why this matters.** ViT papers always assume `H % p == 0`. Real datasets don't. The cheapest fix is right/bottom zero padding; more sophisticated options (reflect padding, learned padding tokens) exist but this is the engineering baseline. The unpatchify step would then crop back to `(H, W)` after reconstruction.

**Why observe the failure first?** Letting students *see* the exception message — rather than reading 'rearrange requires divisibility' in a doc — locks in the failure mode. Next time they see `EinopsError: ... shape mismatch`, they'll recognize it as a divisibility issue immediately.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()